# 9.17 選項的文字意義，怎樣變成可比較的分數？


材料寫「要查卡片進度，編號K17；地址已改好」。問「使用者現在要辦什麼？」應選查卡片進度；改問「哪個流程已完成？」則應選地址變更。候選沒換，問題卻改變該找的資訊。[9.11](https://birdhackor.github.io/tiny-perceptron-vlm/9.11.html)先手填分數，現在要看分數可以怎麼算。

前置是[3.3–3.5的Query、Key與縮放點積](https://birdhackor.github.io/tiny-perceptron-vlm/3.3.html)、[9.11–9.12的候選意義與重排條件](https://birdhackor.github.io/tiny-perceptron-vlm/9.11.html)。先把材料、問題編成一份特徵，把候選描述各編成一份特徵；**編碼器**就是把文字轉成這些向量的網路。再用共享的投影矩陣，把前者變成查詢向量query (q)，後者變成供比對的key (k_i)。同一套投影處理每個實質候選，分數可寫成：

\[
s_i=\frac{q^\mathsf{T}k_i}{\sqrt d}。
\]

`d`是兩個向量的共同寬度。分子把對應分量相乘再加總；分母沿用[3.5的尺度調整](https://birdhackor.github.io/tiny-perceptron-vlm/3.5.html)。這裡只取這個比對分數，再對在場答案做softmax；沒有像注意力那樣接著加權取回Value。這種依當前候選給分、指向其中一項的輸出方式常叫**pointer head（指向候選的評分層）**，不必為每個服務另設一個永久的輸出欄位。

下面所有向量、投影及none規則都是**人工示意**。文字名稱只幫我們對照材料，程式沒有讀懂中文，也沒有訓練。先令兩個軸代表本例的卡片、地址線索；真模型須從資料學出有用特徵，軸不一定有這樣的可讀名稱。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import math
import torch

torch.set_num_threads(1)
features = {"card": torch.tensor([1.0, 0.0]), "address": torch.tensor([0.0, 1.0])}
Wq, Wk = torch.eye(2), torch.eye(2)  # 人工投影；真正訓練時可更新
questions = {"現在要辦": torch.tensor([2.0, 0.0]), "已完成": torch.tensor([0.0, 2.0])}
orders = [["card", "address"], ["address", "card"]]


def score(order, h_question, causal=False):
    q = h_question @ Wq
    keys, previous = [], torch.zeros(2)
    for cid in order:
        h = features[cid]
        # 對照：後面的候選若讀到前面候選，它的key也會改。
        keys.append((h + 0.5 * previous if causal else h) @ Wk)
        previous = previous + h
    real_scores = torch.stack(keys) @ q / math.sqrt(q.numel())
    # none問的是整個集合是否有合適項；max不受候選排列影響。
    none_score = 1.0 - real_scores.max()  # 只為示意，並非可靠的已學規則
    scores = torch.cat([real_scores, none_score[None]])
    ids = order + ["none"]
    p = scores.softmax(-1)
    chosen = min(range(len(ids)), key=lambda i: (-float(scores[i]), ids[i]))
    return dict(zip(ids, p.tolist())), ids[chosen]


for name, h in questions.items():
    for order in orders:
        p, chosen = score(order, h)
        print("獨立", name, order, {k: round(v, 4) for k, v in p.items()}, chosen)
for order in orders:
    p, chosen = score(order, questions["現在要辦"], causal=True)
    print("題內因果", order, {k: round(v, 4) for k, v in p.items()}, chosen)
for order in orders:
    print("獨立平手", order, "→", score(order, torch.ones(2))[1])

先追蹤程式中的形狀。`torch.eye(2)`產生單位矩陣`[[1,0],[0,1]]`，所以這次乘上Wq或Wk都保留原向量。`torch.stack(keys)`把兩個長度2的候選向量疊成`[2,2]`，每候選一列；乘q得到兩個點積，`q.numel()`則取q的元素數2，作為d。none原本只有一個數；`none_score[None]`加一軸變成長度1的向量，再用`torch.cat`接到兩個實質分數後面，得到長度3的共同候選軸。

獨立版本問「現在要辦」時，card、address、none的機率約為0.7124、0.1732、0.1145；反轉列表，數值跟著ID搬動。改問「已完成」，則card與address的機率交換。這讓問題的作用可見；但作用是我們手工指定的，沒有證明模型會學到它。兩個實質候選同分的最後兩行都選address，因為程式按固定ID的字母順序處理平手，不依列表位置。

獨立候選不讀其他實質候選，故重排只重排它們的key與分數。none卻需要看集合：本例讀最高實質分數，正式模型也可用平均、最大值、數量這些**對稱彙整**，即換順序仍得到同一摘要的運算。這保障排列關係，沒有保障none判斷正確。增刪候選時，摘要與softmax分母都可能改，機率無須保持原值。

對照的「題內因果」只多了一行：較後的候選特徵加上前面特徵的一半。card在前時，address因而多了卡片分量，機率升到0.2982；反轉後便沒有，address仍為0.1732。兩次都選card，機率已不只跟著ID搬動。**最後同樣用點積，不代表上游向量沒有受排列影響。**

Kev的公開實作把每個候選結尾的表示當key，把最後的`<decide>`表示當query；後者能讀全題候選，題內又遵守因果順序。因此「不同問題隔離」不等於「候選重排等變」。反過來，一起編碼候選也不必然破壞重排：若整個設計採不含列表位置的集合運算，仍可保留排列關係。保證來自輸入、位置、讀取規則與平手規則的合體，不能單靠pointer這個名稱推得。

練習：把`features["card"]`改成`torch.tensor([0., 1.])`，讓兩個候選的人工表示完全一樣。預測獨立版還能分清這兩種意思嗎？執行後，說明為什麼穩定的平手規則只能固定選擇，無法補回表示中已丟掉的差別。

來源：[Kev `PointerHead`固定程式](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/kev/model.py#L205-L223)、[同版How It Works的題目與候選可見關係](https://github.com/jaredpalmer/kev/blob/2074cc44256200bb37540103fa4572901faa10f3/README.md#how-it-works)。本節保重排的獨立分支與none算例是本課的教學設計。
